In [1]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
%cd /content/drive/My Drive

/content/drive/My Drive


In [3]:
# Import libraries
import os
import numpy as np
from getpass import getpass
from dotenv import load_dotenv

load_dotenv()

# Set OpenAI API key
from google.colab import userdata
OPENAI_API_KEY = userdata.get('OPENAI_API_KEY')
os.environ["OPENAI_API_KEY"] = OPENAI_API_KEY
HF_API_KEY = userdata.get('HUGGINGFACE_API_KEY')
os.environ["HUGGINGFACE_API_KEY"] = HF_API_KEY
serper_api_key = userdata.get('SERPER_API_KEY')
os.environ["SERPER_API_KEY"] = serper_api_key
serp_api_key = userdata.get('SERPER_API_KEY')
os.environ["SERPER_API_KEY"] = serp_api_key
print("✓ Setup complete!")

✓ Setup complete!


In [4]:
import gradio as gr
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline

In [5]:
QAmodel = {
    "gpt-nano": "gpt-4.1-nano",
    "gpt-mini": "gpt-4o-mini",
    "gpt-turbo":"gpt-3.5-turbo",
    "llava-1.5": "llava-hf/llava-1.5-7b-hf",
    "llava-1.5.1": "llava-hf/llava-2.0-13b-hf",
    "blip": "Salesforce/blip2-opt-2.7b"
}

In [6]:
!hf auth login

User is already logged in. Use `hf auth login --force` to force re-login.


In [ ]:
!pip show transformers
#!pip install 'transformers=5.4.0'

Name: transformers
Version: 5.0.0
Summary: Transformers: the model-definition framework for state-of-the-art machine learning models in text, vision, audio, and multimodal models, for both inference and training.
Home-page: https://github.com/huggingface/transformers
Author: The Hugging Face team (past and future) with the help of all our contributors (https://github.com/huggingface/transformers/graphs/contributors)
Author-email: transformers@huggingface.co
License: Apache 2.0 License
Location: /usr/local/lib/python3.12/dist-packages
Requires: filelock, huggingface-hub, numpy, packaging, pyyaml, regex, safetensors, tokenizers, tqdm, typer-slim
Required-by: peft, sentence-transformers


In [ ]:
!pip show langchain

Name: langchain
Version: 1.2.14
Summary: Building applications with LLMs through composability
Home-page: https://docs.langchain.com/
Author: 
Author-email: 
License: MIT
Location: /usr/local/lib/python3.12/dist-packages
Requires: langchain-core, langgraph, pydantic
Required-by: 


In [ ]:
!pip show huggingface_hub

Name: huggingface_hub
Version: 1.7.1
Summary: Client library to download and publish models, datasets and other repos on the huggingface.co hub
Home-page: https://github.com/huggingface/huggingface_hub
Author: Hugging Face, Inc.
Author-email: julien@huggingface.co
License: Apache-2.0
Location: /usr/local/lib/python3.12/dist-packages
Requires: filelock, fsspec, hf-xet, httpx, packaging, pyyaml, tqdm, typer, typing-extensions
Required-by: accelerate, datasets, diffusers, gradio, gradio_client, peft, sentence-transformers, timm, tokenizers, torchtune, transformers


Load Required Libraries

In [7]:
!pip install chromadb

YouTubeSearch

In [8]:
!pip install youtube_search

DuckDuckGoSearch

In [ ]:
#!pip install -U langchain-community[duckduckgo]

In [9]:
!pip install -U ddgs

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.4/46.4 kB 332.3 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 4.7 MB/s eta 0:00:00


In [10]:
!pip install -U langchain langchain-openai
!pip install -q langchain langchain-openai langchain-community langchain-text-splitters

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 112.7/112.7 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 88.5/88.5 kB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 169.8/169.8 kB 10.5 MB/s eta 0:00:00
  Attempting uninstall: langgraph-prebuilt
    Found existing installation: langgraph-prebuilt 1.0.8
    Uninstalling langgraph-prebuilt-1.0.8:
      Successfully uninstalled langgraph-prebuilt-1.0.8
  Attempting uninstall: langgraph
    Found existing installation: langgraph 1.1.4
    Uninstalling langgraph-1.1.4:
      Successfully uninstalled langgraph-1.1.4
  Attempting uninstall: langchain
    Found existing installation: langchain 1.2.14
    Uninstalling langchain-1.2.14:
      Successfully uninstalled langchain-1.2.14
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 27.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 50.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.9/64.9 kB 4.7 MB/s eta 0:00:0

In [11]:
from langchain_community.document_loaders import CSVLoader

# Import medreason instruction dataset
loader = CSVLoader('medreason-instruction-dataset.csv')
docs = loader.load()

print(f"Loaded {len(docs)} document(s)")
print(f"\nTotal characters: {len(docs[0].page_content):,}")
print(f"\nFirst 500 characters:\n{docs[0].page_content[:500]}...")
print(f"\nMetadata: {docs[0].metadata}")

Loaded 31535 document(s)

Total characters: 67

First 500 characters:
query: Most sensitive test for H pylori
answer: D. Urea breath test...

Metadata: {'source': 'medreason-instruction-dataset.csv', 'row': 0}


In [12]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Create text splitter
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,            # Maximum chunk size (characters), reduced for more granularity
    chunk_overlap=100,          # Overlap between chunks (20% of chunk_size)
    add_start_index=True,       # Track position in original document
    separators=["\n\n", "\n", " ", ""]  # Try separators in order
)

# Split the loaded documents
all_splits = text_splitter.split_documents(docs)

print(f"Split each document into {len(all_splits)} chunks")
print(f"\nChunk 0 length: {len(all_splits[0].page_content)} characters")
print(f"Chunk 0 metadata: {all_splits[0].metadata}")
print(f"\nFirst chunk content:\n{all_splits[0].page_content}")

Split each document into 35056 chunks

Chunk 0 length: 67 characters
Chunk 0 metadata: {'source': 'medreason-instruction-dataset.csv', 'row': 0, 'start_index': 0}

First chunk content:
query: Most sensitive test for H pylori
answer: D. Urea breath test


Create embeddings and store them in a vector database.Create a vector store with Chroma.

In [ ]:
from langchain_openai import OpenAIEmbeddings
from langchain_community.vectorstores import Chroma
from langchain_community.embeddings import HuggingFaceEmbeddings # Import HuggingFace embeddings

# Initialize embedding model
# Original: embedding_model = OpenAIEmbeddings(model="text-embedding-3-small")
# Change to HuggingFace embeddings for better retrieval for this dataset
embedding_model = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

# Create vector store from document chunks
persist_directory = './chroma_db_hf_rerun2' # Using a new directory to avoid conflicts and re-embed

vectordb = Chroma.from_documents(
    documents=all_splits,
    embedding=embedding_model,
    persist_directory=persist_directory
)

print(f"✓ Vector store created with {vectordb._collection.count()} document chunks using HuggingFace embeddings")
print(f"✓ Persisted to: {persist_directory}")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


✓ Vector store created with 35056 document chunks using HuggingFace embeddings
✓ Persisted to: ./chroma_db_hf_rerun2


Load existing vector store with the embeddings.

In [13]:
from langchain_community.vectorstores import Chroma
from langchain_community.embeddings import HuggingFaceEmbeddings

# Initialize embedding model with the same model used for creation (sentence-transformers/all-MiniLM-L6-v2)
embedding_model = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

# Set the persist directory to the existing HuggingFace database path
persist_directory = './chroma_db_hf_rerun2'

# Load existing vector store (no need to re-embed)
vectordb = Chroma(
    persist_directory=persist_directory,
    embedding_function=embedding_model
)

print(f"✓ Loaded vector store with {vectordb._collection.count()} documents using HuggingFace embeddings")

/tmp/ipykernel_36562/3739948877.py:5: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embedding_model = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

/tmp/ipykernel_36562/3739948877.py:11: LangChainDeprecationWarning: The class `Chroma` was deprecated in LangChain 0.2.9 and will be removed in 1.0. An updated version of the class exists in the `langchain-chroma package and should be used instead. To use it run `pip install -U `langchain-chroma` and import as `from `langchain_chroma import Chroma``.
  vectordb = Chroma(


✓ Loaded vector store with 35056 documents using HuggingFace embeddings


In [14]:
!pip install -U openai-whisper

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 803.2/803.2 kB 11.1 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 188.3/188.3 MB 5.9 MB/s eta 0:00:00
  Created wheel for openai-whisper: filename=openai_whisper-20250625-py3-none-any.whl size=803979 sha256=a2c50581b027692872a3bff632015620fe1a1c269bfdc24bd5be022131c1654d
  Stored in directory: /root/.cache/pip/wheels/61/d2/20/09ec9bef734d126cba375b15898010b6cc28578d8afdde5869
Successfully built openai-whisper


In [15]:
# Load whisper
import whisper

# Load the Whisper model globally for efficiency within the Gradio app
whisper_model = whisper.load_model("turbo")

100%|██████████████████████████████████████| 1.51G/1.51G [00:06<00:00, 253MiB/s]


In [16]:
# Install triton
!pip install -q triton

In [17]:
# Install accelerate
!pip install accelerate

In [18]:
# Install bitsandbytes with the required version
!pip install -q bitsandbytes>=0.46.1

In [19]:
!pip install -q -U huggingface_hub

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 642.6/642.6 kB 9.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.2/4.2 MB 83.4 MB/s eta 0:00:00


In [ ]:
# Restart the runtime to ensure new library versions are loaded. Hopefully this is npt needed but it gets the job done.
#import os
#os.kill(os.getpid(), 9)

In [20]:
# ============================================
# LOAD A BLIP2 PROCESSOR/MODEL
# ============================================
from PIL import Image
import httpx
from io import BytesIO
from transformers import Blip2Processor, Blip2ForConditionalGeneration
import torch

model_id = QAmodel['blip']
device = "cuda" if torch.cuda.is_available() else "cpu"

blip2_model = Blip2ForConditionalGeneration.from_pretrained(model_id, dtype=torch.float16).to(device)  # doctest: +IGNORE_RESULT

blip2_processor = Blip2Processor.from_pretrained(model_id)



config.json: 0.00B [00:00, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/1247 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/141 [00:00<?, ?B/s]

...

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

added_tokens.json:   0%|          | 0.00/23.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/548 [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Preparing the quantization config to load the model in 4bit precision

In [21]:
import torch
from transformers import BitsAndBytesConfig

# Determine device and dtype based on CUDA availability
if torch.cuda.is_available():
    device = 'cuda'
    dtype = torch.float16
    print("Using GPU for model loading.")
    quantization_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_compute_dtype=torch.float16
    )
else:
    device = 'cpu'
    dtype = torch.float32 # float16 generally not supported or efficient on CPU
    print("Using CPU for model loading as no GPU is available.")
    quantization_config = None # No quantization for CPU

Using CPU for model loading as no GPU is available.


In [ ]:
#import requests
#from PIL import Image
#import torch
#from transformers import AutoProcessor, LlavaForConditionalGeneration

# ============================================
# LOAD A Llava PROCESSOR
# ============================================
#model_id = QAmodel['llava-1.5']

# Load model based on device availability
#if device == 'cuda':
#    llava_model = LlavaForConditionalGeneration.from_pretrained(
#        model_id,
#        torch_dtype=dtype,
#        low_cpu_mem_usage=True,
#        quantization_config=quantization_config
#    ).to(device)
#else:
    # Load without quantization for CPU
#    llava_model = LlavaForConditionalGeneration.from_pretrained(
#        model_id,
#        torch_dtype=dtype,
#        low_cpu_mem_usage=True
#    ).to(device)

#llava_processor = AutoProcessor.from_pretrained(model_id)

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/686 [00:00<?, ?it/s]

The image processor of type `CLIPImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. 


In [38]:
from langchain_openai import ChatOpenAI
# LLM-powered topic classifier model
classifier_model = ChatOpenAI(model=QAmodel['gpt-mini'], temperature=0)
classifier_topic = ChatOpenAI(model=QAmodel['gpt-nano'], temperature=0)

In [27]:
# Introduce global flags to track state across agent invocations
global _kb_had_results
_kb_had_results = False
global is_medical_query
is_medical_query = False # Initialize globally

In [28]:
# =======================================
# Global Tool Definitions
# =======================================
from langchain.tools import tool

@tool
def calculate_bmi(weight_kg: float, height_m: float) -> str:
    """Calculates Body Mass Index (BMI) given weight in kilograms and height in meters.
    Use this tool when the user asks for BMI and provides both weight and height.
    Example: calculate_bmi(weight_kg=70, height_m=1.75)
    """
    if height_m <= 0 or weight_kg <= 0:
        return "Error: Height and weight must be positive values."
    bmi = weight_kg / (height_m ** 2)
    if bmi < 18.5:
        category = "Underweight"
    elif 18.5 <= bmi < 24.9:
        category = "Normal weight"
    elif 25 <= bmi < 29.9:
        category = "Overweight"
    else:
        category = "Obesity"
    return f"Your BMI is {bmi:.2f}, which falls into the '{category}' category."

@tool
def retrieve_context(query: str) -> str:
    """Retrieve relevant context from knowledge base for a question."""
    global _kb_had_results # Declare intent to modify global variable
    try:
        # Ensure vectordb is accessible (it's loaded globally)
        docs = vectordb.max_marginal_relevance_search(query, k=1, fetch_k=1)
        if not docs:
            _kb_had_results = False
            return "No relevant information found in knowledge base."
        else:
            _kb_had_results = True
            print(f"Retrieved {len(docs)} document(s)")
            return "\n\n".join([f"Source {i+1}:\n{doc.page_content[:500]}..." for i, doc in enumerate(docs)])
    except Exception as e:
        _kb_had_results = False
        print(f"Error in retrieve_context tool: {e}")
        return f"Error retrieving context from knowledge base: {e}"

@tool
def youtube_links_tool(query: str) -> str:
    """Search YouTube and return best relevant medical explanation video links."""
    print(f"Executing youtube_links_tool with query: {query}")
    try:
        youtube_tool_instance = YouTubeSearchTool(top_k=3,description="Search for YouTube videos and return top unique medical video links.")
        video_urls = youtube_tool_instance.invoke({'query': query})
        print(f"Raw YouTube search output: {video_urls[:200]}...") # Print first 200 chars for brevity

        if not video_urls:
            return "No relevant YouTube videos found."

        output = "Top YouTube videos:\n"
        for url_string in video_urls:
            match = re.search(r"v=([a-zA-Z0-9_-]+)", url_string)
            video_id = match.group(1) if match else "N/A"
            title = f"YouTube Video (ID: {video_id})" # Placeholder title

            output += f"- {url_string}\n" # Use the full URL string as the link

        return output
    except Exception as e:
        print(f"Error in youtube_links_tool: {e}")
        return f"Error searching YouTube: {e}"

@tool
def create_calendar_event(
    title: str,
    start_time: str,       # ISO format: "2024-01-15T14:00:00"
    end_time: str,         # ISO format: "2024-01-15T15:00:00"
    attendees: list[str],  # email addresses
    location: str = ""
) -> str:
    """Create a calendar event. Requires exact ISO datetime format."""
    # Stub: In practice, this would call Google Calendar API, Outlook API, etc.
    return f"Event created: {title} from {start_time} to {end_time} with {len(attendees)} attendees"

@tool
def send_email(
    to: list[str],  # email addresses
    subject: str,
    body: str,
    cc: list[str] = []
) -> str:
    """Send an email via email API. Requires properly formatted addresses."""
    # Stub: In practice, this would call SendGrid, Gmail API, etc.
    return f"Email sent to {', '.join(to)} - Subject: {subject}"

@tool
def get_available_time_slots(
    attendees: list[str],
    date: str,  # ISO format: "2024-01-15"
    duration_minutes: int
) -> list[str]:
    """Check calendar availability for given attendees on a specific date."""
    # Stub: In practice, this would query calendar APIs
    return ["09:00", "14:00", "16:00"]

Create an Email Agent

In [29]:
EMAIL_AGENT_PROMPT = (
    "You are an email assistant. "
    "Compose professional emails based on natural language requests. "
    "Extract recipient information and craft appropriate subject lines and body text. "
    "Use send_email to send the message. "
    "Always confirm what was sent in your final response."
)

email_agent = create_agent(
    classifier_model,
    tools=[send_email],
    system_prompt=EMAIL_AGENT_PROMPT,
)

Test the Email Agent

In [47]:
query = "Send the design team a reminder about reviewing the new mockups"

for step in email_agent.stream(
    {"messages": [{"role": "user", "content": query}]}
):
    for update in step.values():
        for message in update.get("messages", []):
            message.pretty_print()

================================== Ai Message ==================================
Tool Calls:
  send_email (call_tU9Usxp3CFX7WdgUvdZxP3tX)
 Call ID: call_tU9Usxp3CFX7WdgUvdZxP3tX
  Args:
    to: ['designteam@example.com']
    subject: Reminder: Review of New Mockups
    body: Dear Design Team,

This is a friendly reminder to review the new mockups that were shared last week. Your feedback is crucial for the next steps in our project.

Please make sure to provide your thoughts by the end of the week.

Thank you!

Best regards,
[Your Name]
================================= Tool Message =================================
Name: send_email

Email sent to designteam@example.com - Subject: Reminder: Review of New Mockups
================================== Ai Message ==================================

I have sent the email to the design team with the subject "Reminder: Review of New Mockups." The message includes a reminder for them to review the new mockups and provide feedback by the end of t

Install ipdb for local debugging

In [30]:
!pip install ipdb

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 17.3 MB/s eta 0:00:00


In [43]:
import ipdb

In [42]:
%pdb on

Automatic pdb calling has been turned ON


In [31]:
system_prompt=(
    "You are a helpful Medical Q&A assistant. "
    "For any medical question, you MUST use the 'retrieve_context' tool call first to search the knowledge base. "
    "If the knowledge base does not provide a sufficient answer, then you MUST use youtube search tool calls. "
    "For medical question, include youtube video links with response. Keep the answer short and concise. "
    "Use three sentences maximum from all tools. "
    "If question contains keyword 'BMI', you MUST only use the 'calculate_bmi' tool call. "
)

In [32]:
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain.tools import tool
from langchain.agents import create_agent
from langchain.agents.middleware import before_model, AgentState, AgentMiddleware, LLMToolSelectorMiddleware, ToolRetryMiddleware
from langchain.agents.middleware import (
    TodoListMiddleware,      # Task planning
    SummarizationMiddleware, # Compress long convos
    HumanInTheLoopMiddleware
)
from langchain_community.vectorstores import Chroma
from langchain_community.tools import YouTubeSearchTool, DuckDuckGoSearchRun
from langchain_core.messages import AIMessage, ToolMessage
from typing import Any, Dict
from langgraph.runtime import Runtime
import os, json
import re
import httpx # Added for ToolRetryMiddleware
from google.colab import userdata # Moved import here to ensure userdata is defined

# Set your OpenAI API key as an environment variable
os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")

# =======================================
# Global Tool Definitions
# =======================================
retrieve_context = retrieve_context
youtube_links_tool = youtube_links_tool
calculate_bmi = calculate_bmi
web_search = DuckDuckGoSearchRun(max_results=3)

tools = [retrieve_context, youtube_links_tool, calculate_bmi, web_search]

# ===================================================================================
# Global Middleware Definitions
# ===================================================================================
#@before_model(can_jump_to=["end"])
@before_model()
def medical_classifier(state: AgentState, runtime: Runtime) -> Dict[str, Any] | None:
    """LLM decides if query is medical and applies guardrails."""
    if not state["messages"]:
        return None
    user_query = state["messages"][0].content
    try:

        #query_for_classification = guardrail_output
        class_prompt = f"Is this medical/health/symptoms/healthcare/urgent care related? 'YES' or 'NO' only.\nQuery: '{user_query}'\nYES: symptoms/treatments. NO: other."
        class_result = classifier_model.invoke([{"role": "user", "content": class_prompt}])
        print(f"[Medical Classifier] User Query: '{user_query}' -> Classifier Result: '{class_result.content}'") # Debug print

        #if "YES" not in class_result.content.upper():
        #    is_medical_query = False # Set to False if not medical
        #    return {"messages": [AIMessage("I apologize, I am programmed to answer medical questions only.")], "jump_to": "end"}

        if "NO" in class_result.content.upper():
            is_medical_query = False # Set to False if not medical
            return {"messages": [AIMessage("I apologize, I am programmed to answer medical questions only.")], "jump_to": "end"}
        else:
            is_medical_query = True # Set to True if medical
    except Exception as e:
        print(f"Error in medical_classifier (guardrail or classification): {e}")
        is_medical_query = False # or True, depending on desired fallback behavior
        return None # Proceed to main agent
    return None # Proceed to main agent

class DisclaimerMiddleware(AgentMiddleware):
    def after_model(self, state: AgentState, runtime: Runtime) -> Dict[str, Any] | None:
        #global _kb_had_results, is_medical_query # Access global flags
        if state["messages"] and isinstance(state["messages"][-1], AIMessage):
          #if "BMI" in content.upper():
          #if is_medical_query and (not _kb_had_results or "web_search" in state["messages"][-1].content.lower()):
          if is_medical_query and (not _kb_had_results):
              content = state["messages"][-1].content
              state["messages"][-1].content = f"According to medical websites, including MedlinePlus, Cleveland Clinic and Mayo Clinic, {content}"
        _kb_had_results = False
        return state

synthesizer = SummarizationMiddleware(
    model=classifier_model,
    trigger=("tokens", 4000),
    max_tokens=1000,
    system_prompt="Synthesize key points, action items, and recent context.",
)

toolselector = LLMToolSelectorMiddleware(model=QAmodel.get('gpt-mini'),system_prompt="",max_tools=5)
toolretry = ToolRetryMiddleware(max_retries=3,backoff_factor=2.0,retry_on=[TimeoutError,httpx.NetworkError])

# Global Agent Setup
agent = create_agent(
    classifier_model,
    tools=tools,
    system_prompt=system_prompt,
    middleware=[
        medical_classifier,
        #TodoListMiddleware(),
        ##synthesizer,
        #toolselector,
        #toolretry #,
        DisclaimerMiddleware()
    ],
)

# =========================
# GENERATE-CHAT FUNCTION (Invokes the agent created using LangChain)
# =========================
def generate_chat(user_input, chat_history): # Removed model_choice parameter

  #_kb_had_results = False
  #is_medical_query = False

  # Prepare messages for the agent (chat_history is now expected to be list of dictionaries)
  agent_messages = list(chat_history) # Create a mutable copy of the existing chat history
  agent_messages.append({"role": "user", "content": user_input}) # Add the current user's message

  # Invoke the agent
  response = agent.invoke({"messages": agent_messages})

  # Extract retrieved_answer from tool messages (using 'response')
  retrieved_answer = None
  for message in response['messages']:
    if isinstance(message, ToolMessage) and message.name == 'retrieve_context':
        content_lines = message.content.split('\n')
        for line in content_lines:
            if line.strip().startswith('answer:'):
                retrieved_answer = line.strip().replace('answer: ', '') # Extract just the answer text
                break # Found the answer, break from inner loop
        if retrieved_answer:
            break # Found the answer, break from outer loop

  if retrieved_answer:
    print(f"Direct answer from retrieve_context tool: '{retrieved_answer}'")
  else:
    # Only print final AI message if no direct tool answer was extracted for console clarity
    pass # No need to print here, final_ai_message below will capture the response.

  # Extract the final AI message content
  final_ai_message = "I cannot find the best answer. Please consult with a Doctor." # Default fallback
  for msg in reversed(response['messages']):
      if isinstance(msg, AIMessage):
          final_ai_message = msg.content
          break

  # Append the new user input and agent response to the chat history
  # The output should also be a list of dictionaries, consistent with gr.Chatbot(type='messages')
  updated_chat_history = list(chat_history) # Start with existing chat history
  updated_chat_history.append({"role": "user", "content": user_input}) # Add current user message
  updated_chat_history.append({"role": "assistant", "content": final_ai_message}) # Add assistant's final response

  return updated_chat_history

In [33]:
  # =======================================
  # Function to transcribe audio to text
  # =======================================
def speech_to_text(audio_file_path, chat_history): # Removed model_choice parameter
    if audio_file_path is None:
        return "", chat_history
    try:
        # Ensure the audio file exists and is accessible
        result = whisper_model.transcribe(audio_file_path)
        transcribed_text = result.get("text", "").strip()

        # Process the transcribed text through the chat logic
        if not transcribed_text:
            return "", chat_history # Return original chat history if transcription is empty

        # Call the main chat generation function with the transcribed text
        new_chat_history = generate_chat(transcribed_text, chat_history) # Removed model_choice

        # Return an empty string for the text input and the updated chat history
        return "", new_chat_history

    except Exception as e:
        print(f"Error during audio transcription: {e}")
        # Return an error message to the user, appending it to chat_history
        error_message = f"Error transcribing audio: {e}"
        if chat_history is None:
            chat_history = []
        chat_history.append({"role": "user", "content": "(Audio input failed)"})
        chat_history.append({"role": "assistant", "content": error_message})
        return "", chat_history


In [34]:
import requests
from PIL import Image
import torch

  # =======================================
  # Function to transform OCR to text
  # =======================================
def image_to_text_processor(image_path, processor_obj, model_obj, device_obj, dtype_obj, vtmodel_name):
    if image_path is None:
        return "Please upload an image first."

    try:
        image = Image.open(image_path).convert("RGB") # Ensure RGB conversion for consistency

        # Prepare the prompt as required by the model
        if vtmodel_name == QAmodel['blip']:
            prompt_text = "Question: What do you see in the image? Answer:"
        else: # LLaVA
            prompt_text = "USER: <image>\nWhat do you see in the image?\nASSISTANT:"

        # Process the image and prompt using the passed processor object
        inputs = processor_obj(images=image, text=prompt_text, return_tensors='pt').to(device_obj, dtype_obj)

        # Generate output using the passed model object
        output = model_obj.generate(**inputs, max_new_tokens=200, do_sample=False)

        # Decode the output using the passed processor object
        if vtmodel_name == QAmodel['blip']:
            outputs = processor_obj.batch_decode(output, skip_special_tokens=True)[0].strip()
            # BLIP's raw output usually contains the prompt as well, extract just the answer
            if "Answer:" in outputs:
                final_output = outputs.split("Answer:", 1)[1].strip()
            else:
                final_output = outputs.strip() # Fallback
        else: # llava
            outputs = processor_obj.decode(output[0][2:], skip_special_tokens=True)
            # Extract only the assistant's answer from the decoded output
            assistant_prefix = "ASSISTANT:"
            if assistant_prefix in outputs:
                answer_start_index = outputs.find(assistant_prefix) + len(assistant_prefix)
                final_output = outputs[answer_start_index:].strip()
            else:
                final_output = outputs.strip() # Fallback if prefix is not found

        return final_output
    except Exception as e:
        return f"Error processing image by {vtmodel_name} processor: {e}"

Gradio UI

In [35]:
import gradio as gr
from functools import partial # Import functools
import os # Import os for path checking

with gr.Blocks(theme=gr.themes.Soft()) as demo:
    gr.Markdown(
        """
        <style>
            .gradio-container {
                border: 2px solid #a8dadc; /* Light blue/teal border */
                border-radius: 10px; /* Slightly rounded corners */
                box-shadow: 0 4px 8px rgba(0, 0, 0, 0.1); /* Subtle shadow */
                padding: 15px; /* Add some padding inside the border */
            }
            /* Target the chat window specifically */
            .gr-chatbot {
                border: 1px solid #d1e2ec; /* Lighter border for chatbox itself */
                border-radius: 8px;
                border-top: 3px solid #6c757d; /* Stronger top border for separation */
            }
        </style>
        <h1 style='text-align: center; margin-bottom: 1em;'>
            <img src='https://upload.wikimedia.org/wikipedia/commons/thumb/1/18/Red_cross_icon_%28vector%29.svg/1200px-Red_cross_icon_%28vector%29.png' alt='' style='height: 10px; vertical-align: middle; margin-right: 10px;'>
            MedIntel Q&A Assistant
        </h1>
        <p style='text-align: center; font-size: 1.1em; color: #555;'>
            Ask medical questions and get answers with knowledge base retrieval, LLM generation, and YouTube references.
        </p>
        """
    )

    with gr.Row():
      model_choice = QAmodel['gpt-mini'] # This is currently set to gpt-mini
      vtmodel_id = QAmodel['blip'] # Set default VT model to BLIP. User can change this in future.

    with gr.Column(scale=4):
        chatbot = gr.Chatbot(label="Conversation History", height=350, type='messages', allow_tags=False) # Added type and allow_tags
        txt_input = gr.Textbox(show_label=False, placeholder="Type your medical question here...", lines=1)

        with gr.Row():
            # Changed from gr.File to gr.UploadButton
            upload_file = gr.UploadButton("Upload Image", type="filepath")
            image_to_text_btn = gr.Button("Image-to-Text", variant="secondary")

        with gr.Row():
            audio_input = gr.Audio(sources=['microphone'], type="filepath", label="Record your question", scale=1)
            transcribe_button = gr.Button("Transcribe Audio & Submit", variant="secondary", scale=1)

        with gr.Row():
            submit_btn = gr.Button("Submit Question", variant="primary", scale=1)
            clear_btn = gr.ClearButton(value="Clear Chat", scale=0)

    def respond(user_input, chat_history):
        try:
            if not user_input:
                return "", chat_history # Don't process empty input

            if chat_history is None:
                chat_history = []
            #breakpoint()
            #is_medical_query = False # Reset to False at the start of each chat
            new_chat_history = generate_chat(user_input, chat_history)

            return "", new_chat_history
        except Exception as e:
            error_message = f"An error occurred: {e}"
            print(f"Error in respond function: {e}") # Print to Colab console

            if chat_history is None:
                chat_history = []
            updated_chat_history = list(chat_history) # Make a copy to append
            updated_chat_history.append({"role": "user", "content": user_input})
            updated_chat_history.append({"role": "assistant", "content": error_message})
            return "", updated_chat_history # Return the updated history

    def process_image_for_chat(image_file_path_str, chat_history):
        print(f"DEBUG: Received image_file_path_str from Gradio: {image_file_path_str}") # Debug print

        image_path = image_file_path_str

        print(f"DEBUG: Processed image_path: {image_path}") # Debug print

        if image_path is None or not isinstance(image_path, str) or not os.path.isfile(image_path):
            error_message = f"Error: Provided path is not a valid image file: {image_path}. Please ensure a file is uploaded."
            print(f"DEBUG: {error_message}")
            if chat_history is None:
                chat_history = []
            updated_chat_history = list(chat_history)
            updated_chat_history.append({"role": "assistant", "content": error_message})
            return updated_chat_history, None # Return current chat and clear image if no image uploaded

        if chat_history is None:
            chat_history = []

        user_message_content = "What do you see in the image?"
        updated_chat_history = list(chat_history)
        updated_chat_history.append({"role": "user", "content": user_message_content})

        try:
            # Dynamically select the correct processor and model objects based on vtmodel_id
            selected_processor = None
            selected_model = None
            global vtmodel_id # Ensure vtmodel_id is accessible

            if vtmodel_id == QAmodel['blip']:
                selected_processor = blip2_processor
                selected_model = blip2_model
            elif vtmodel_id == QAmodel['llava-1.5']:
                selected_processor = llava_processor
                selected_model = llava_model
            else:
                image_description = f"Unsupported vision-transformer model ID: {vtmodel_id}."
                updated_chat_history.append({"role": "assistant", "content": image_description})
                return updated_chat_history, None

            # Call the unified image_to_text_processor with the selected components
            image_description = image_to_text_processor(image_path, selected_processor, selected_model, device, dtype, vtmodel_id)
            updated_chat_history.append({"role": "assistant", "content": image_description})
        except Exception as e:
            error_message = f"Error processing image: {e}"
            print(f"Error in process_image_for_chat: {e}")
            updated_chat_history.append({"role": "assistant", "content": error_message})

        return updated_chat_history, None # Return updated chat history and clear the image input

    transcribe_button.click(
        speech_to_text,
        inputs=[audio_input, chatbot],
        outputs=[txt_input, chatbot]
    )

    submit_btn.click(respond, [txt_input, chatbot], [txt_input, chatbot])
    txt_input.submit(respond, [txt_input, chatbot], [txt_input, chatbot])
    clear_btn.click(lambda: [], None, chatbot, queue=False) # Clear chatbot on button click, return empty list

    image_to_text_btn.click(
        process_image_for_chat,
        inputs=[upload_file, chatbot],
        outputs=[chatbot, upload_file]
    )

/tmp/ipykernel_36562/3552901238.py:5: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(theme=gr.themes.Soft()) as demo:
/tmp/ipykernel_36562/3552901238.py:37: DeprecationWarning: The default value of 'allow_tags' in gr.Chatbot will be changed from False to True in Gradio 6.0. You will need to explicitly set allow_tags=False if you want to disable tags in your chatbot.
  chatbot = gr.Chatbot(label="Conversation History", height=350, type='messages', allow_tags=False) # Added type and allow_tags


In [36]:
demo.close()

Launch the interface

In [37]:
demo.launch()

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://26c1cf36c48233e4d1.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Testing

In [25]:
from langchain.tools import tool

@tool
def calculate_bmi(weight_kg: float, height_m: float) -> str:
    """Calculates Body Mass Index (BMI) given weight in kilograms and height in meters.
    Use this tool when the user asks for BMI and provides both weight and height.
    Example: calculate_bmi(weight_kg=70, height_m=1.75)
    """
    if height_m <= 0 or weight_kg <= 0:
        return "Error: Height and weight must be positive values."
    bmi = weight_kg / (height_m ** 2)
    if bmi < 18.5:
        category = "Underweight"
    elif 18.5 <= bmi < 24.9:
        category = "Normal weight"
    elif 25 <= bmi < 29.9:
        category = "Overweight"
    else:
        category = "Obesity"
    return f"Your BMI is {bmi:.2f}, which falls into the '{category}' category."

In [26]:
from langchain.agents import create_agent
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain.agents.middleware import before_model, AgentState, AgentMiddleware, LLMToolSelectorMiddleware, ToolRetryMiddleware, TodoListMiddleware, SummarizationMiddleware # Import SummarizationMiddleware
from typing import Any, Dict
from langgraph.runtime import Runtime

# Reset _kb_had_results for each new query invocation
global _kb_had_testresults
_kb_had_testresults = False

# LLM-powered topic classifier middleware
classifier_modeltest = ChatOpenAI(model=QAmodel['gpt-mini'], temperature=0, streaming=True, openai_api_key=userdata.get("OPENAI_API_KEY"))
classifier_topictest = ChatOpenAI(model=QAmodel['gpt-nano'], temperature=0, streaming=True, openai_api_key=userdata.get("OPENAI_API_KEY"))

# Initialize global flag for testing
global is_medical_querytest
is_medical_querytest = False

# =======================================
# Summarization middleware (instantiate directly)
# =======================================
synthesizertest = SummarizationMiddleware(
    model=classifier_modeltest,           # Pass the classifier_model as the LLM for summarization
    trigger=("tokens", 4000),         # Compress when history > 4000 tokens
    max_tokens=1000,                  # Max length of summary
    summary_prompt="Synthesize key points, action items, and recent context.",
    system_prompt="Synthesize key points, action items, and recent context.",  # Custom instructions
)

# Initialize the LLM for the agent (using gpt-4.1-nano as per notebook's preference)
# Ensure OPENAI_API_KEY is set in your environment or Google Colab userdata
llm_agenttest = ChatOpenAI(model="gpt-4.1-nano", temperature=0)

# Define the tools the agent can use
medical_toolstest = [calculate_bmi]

# ===================================================================================
# Medical classifier - Runs before the main agent model tries to answer the question.
# ===================================================================================
@before_model(can_jump_to=["end"])
def medical_classifiertest(state: AgentState, runtime: Runtime) -> Dict[str, Any] | None:
    """LLM decides if query is medical and applies guardrails."""
    global is_medical_querytest # Declare intent to modify global variable
    if not state["messages"]:
        return None
    user_query = state["messages"][0].content
    try:
      print(f"Entering medical_classifiertest")

      # Apply the guardrail first
      # Assuming execute_chat_with_guardrail returns a string:
      # - The original/sanitized query if safe to proceed.
      # - A rejection message if unsafe.
      #guardrail_output = execute_chat_with_guardrail(user_query)

      # Check if the guardrail returned a rejection message (heuristic)
      #if "cannot process" in guardrail_output.lower() or "safety concerns" in guardrail_output.lower():
      #    is_medical_query = False # Treat as rejected
      #    return {"messages": [AIMessage(guardrail_output)], "jump_to": "end"}

      # If guardrail_output is not a rejection, then it's the (potentially sanitized) query.
      # Proceed with medical classification using this output.
      #query_for_classification = guardrail_output

      class_prompt = f"Is this medical/health/symptoms/healthcare/urgent care related? 'YES' or 'NO' only.\nQuery: '{user_query}'\nYES: symptoms/treatments. NO: other."
      class_result = classifier_topictest.invoke([{"role": "user", "content": class_prompt}])
      print(f"[Medical Classifier] User Query: '{user_query}' -> Classifier Result: '{class_result.content}'") # Debug print
      if "NO" in class_result.content.upper():
          is_medical_querytest = False # Set to False if not medical
          return {"messages": [AIMessage("I apologize, I am programmed to answer medical questions only.")], "jump_to": "end"}
      else:
          is_medical_querytest = True # Set to True if medical
    except Exception as e:
        print(f"Error in medicaltopic_classifier : {e}")
        # If guardrail or classifier fails, default to medical to let the agent attempt to answer, or return an error
        is_medical_querytest = True # or False, depending on desired fallback behavior
        return None # Proceed to main agent
    return None # Proceed to main agent

# =======================================
# Custom middleware for adding disclaimer
# =======================================
class DisclaimerMiddleware(AgentMiddleware): # Inherit from AgentMiddleware
    def after_model(self, state: AgentState, runtime: Runtime) -> Dict[str, Any] | None:
        global _kb_had_testresults, is_medical_querytest # Access global flags
        # Check if the last message is an AIMessage (final response from the agent)
        if state["messages"] and isinstance(state["messages"][-1], AIMessage):
            # Add disclaimer if it's a medical query AND
            # either web_search was used (implied by "web_search" in str(state["messages"][-1].content)))
            # OR KB returned no results (_kb_had_results is False)
            # Note: "web_search" in content might not be reliable, rely more on _kb_had_results
            if is_medical_querytest and (not _kb_had_testresults or "web_search" in state["messages"][-1].content.lower()):
                content = state["messages"][-1].content
                state["messages"][-1].content = f"According to medical websites, including MedlinePlus, Cleveland Clinic and Mayo Clinic, {content}"
        # Reset _kb_had_results for the next invocation
        _kb_had_testresults = False
        return state

# Create the agent using create_agent
#agent = create_agent(llm_agent, medical_tools, agent_prompt)
system_prompt="You are a helpful Medical Q&A assistant. For any medical question, you MUST use the 'retrieve_context' tool first to search the knowledge base. If the knowledge base does not provide a sufficient answer, then you MUST use web search and youtube search tools. For medical question, include youtube video links with web search results.Keep the answer short and concise. Use three sentences maximum based on the information gathered from all the tools. Don't use Disclaimer if the question contains 'BMI' keyword in it.",


agenttest = create_agent(
    llm_agenttest,
    medical_toolstest,
    system_prompt="You are a helpful Medical Q&A assistant. For any medical question, you MUST use the 'retrieve_context' tool first to search the knowledge base. If the knowledge base does not provide a sufficient answer, then you MUST use web search and youtube search tools. For medical question, include youtube video links with web search results.Keep the answer short and concise. Use three sentences maximum based on the information gathered from all the tools. Don't use Disclaimer if the question contains 'BMI' keyword in it.",
    middleware=[
    medical_classifiertest,
    synthesizertest,
        #after_model
    DisclaimerMiddleware()]
    )

# Create an agent executor to run the agent
#agent_executor = AgentExecutor(agent=agent, tools=medical_tools, verbose=True)

In [120]:
print("\n--- Testing medcal classifier ---")

try:
  #result = agenttest.invoke({"messages": [{"role": "user", "content": "How can I cut grass?"}]})
  #result = agenttest.invoke({"messages": [{"role": "user", "content": "How can I repair microwave oven?"}]})
  #result = agenttest.invoke({"messages": [{"role": "user", "content": "How can I lower my electricity bill?"}]})
  #result = agenttest.invoke({"messages": [{"role": "user", "content": "How can I mop the floor?"}]})
  #result = agenttest.invoke({"messages": [{"role": "user", "content": "How can I wash clothes?"}]})
  #result = agenttest.invoke({"messages": [{"role": "user", "content": "Can you tell me what time it is now?"}]})
  #result = agenttest.invoke({"messages": [{"role": "user", "content": "Is the sky blue?"}]})
  #result = agenttest.invoke({"messages": [{"role": "user", "content": "Where is my toothpaste?"}]})

  result = agenttest.invoke({"messages": [{"role": "user", "content": "what should I do if I'm experiencing a back pain?"}]})
  print(result["messages"][-1].content)

  #response = agent.invoke({"input": "How can I make explosives?", "agent_scratchpad": []})
  #print(f"\nAgent Response (BMI valid): {response['messages'][-1].content}")

except Exception as e:
    print(f"Error running medcal classifier: {e}")


--- Testing medcal classifier ---
Entering medical_classifiertest
[Medical Classifier] User Query: 'what should I do if I'm experiencing a back pain?' -> Classifier Result: 'YES'
Entering medical_classifiertest
[Medical Classifier] User Query: 'what should I do if I'm experiencing a back pain?' -> Classifier Result: 'YES'
According to medical websites, including MedlinePlus, Cleveland Clinic and Mayo Clinic, If you're experiencing back pain, rest and avoid strenuous activities, apply ice or heat, and consider over-the-counter pain relievers. If the pain persists for more than a few days, worsens, or is accompanied by other symptoms like numbness or weakness, consult a healthcare professional. Proper posture and gentle stretching may also help prevent future episodes.


In [37]:
#%debug

> /tmp/ipykernel_65017/1131487107.py(9)<cell line: 0>()
      7 )
      8 
----> 9 email_agent = create_agent(
     10     classifier_model,
     11     tools=[send_email],

ipdb> _kb_had_results
True
ipdb> is_medical_query
False
[Medical Classifier] User Query: 'I weigh 70 kilograms and my height is 1.75 meters. What is my BMI?' -> Classifier Result: 'YES'
ipdb> is_medical_query
False
--KeyboardInterrupt--

KeyboardInterrupt: Interrupted by user
